In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "source": [
    "# RNN for Text Generation\n",
    "Train a Recurrent Neural Network to generate Shakespeare-style text."
   ]
  },
  {
   "cell_type": "code",
   "source": [
    "import tensorflow as tf\n",
    "import numpy as np\n",
    "import os, time\n",
    "path_to_file = tf.keras.utils.get_file('shakespeare.txt',\n",
    "    'https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt')"
   ]
  },
  {
   "cell_type": "code",
   "source": [
    "text = open(path_to_file, 'rb').read().decode(encoding='utf-8')\n",
    "vocab = sorted(set(text))\n",
    "char2idx = {u:i for i, u in enumerate(vocab)}\n",
    "idx2char = np.array(vocab)\n",
    "text_as_int = np.array([char2idx[c] for c in text])"
   ]
  },
  {
   "cell_type": "code",
   "source": [
    "# Create training examples\n",
    "seq_length = 100\n",
    "examples_per_epoch = len(text)//(seq_length+1)\n",
    "char_dataset = tf.data.Dataset.from_tensor_slices(text_as_int)\n",
    "sequences = char_dataset.batch(seq_length+1, drop_remainder=True)\n",
    "\n",
    "def split_input_target(chunk):\n",
    "    input_text = chunk[:-1]\n",
    "    target_text = chunk[1:]\n",
    "    return input_text, target_text\n",
    "\n",
    "dataset = sequences.map(split_input_target)"
   ]
  },
  {
   "cell_type": "code",
   "source": [
    "# Create model\n",
    "BATCH_SIZE = 64\n",
    "BUFFER_SIZE = 10000\n",
    "vocab_size = len(vocab)\n",
    "embedding_dim = 256\n",
    "rnn_units = 1024\n",
    "\n",
    "dataset = dataset.shuffle(BUFFER_SIZE).batch(BATCH_SIZE, drop_remainder=True)\n",
    "\n",
    "def build_model(vocab_size, embedding_dim, rnn_units, batch_size):\n",
    "    model = tf.keras.Sequential([\n",
    "        tf.keras.layers.Embedding(vocab_size, embedding_dim,\n",
    "                                  batch_input_shape=[batch_size, None]),\n",
    "        tf.keras.layers.GRU(rnn_units,\n",
    "                           return_sequences=True,\n",
    "                           stateful=True,\n",
    "                           recurrent_initializer='glorot_uniform'),\n",
    "        tf.keras.layers.Dense(vocab_size)\n",
    "    ])\n",
    "    return model\n",
    "\n",
    "model = build_model(vocab_size, embedding_dim, rnn_units, BATCH_SIZE)"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "name": "python3",
   "display_name": "Python 3",
   "language": "python"
  },
  "language_info": {
   "name": "python"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}
